# Practice: evaluating LLMs
*AI from scratch, stage 6* · lesson: https://ihor.sh/ai/evaluating-llms.html

Build the grading side of an eval: a normalizing check, Cohen's kappa, a small model as judge (and how far to trust
it), a position-bias test for pairwise judging, and a paired comparison of SmolLM2 with and without retrieval on 20
questions about this course. A few minutes on a CPU. Replace each `...` under `# YOUR CODE HERE`, run the cell, then
the check under it; `hint(n)` and `answer(n)` are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['" ".join(re.findall(r"[a-z0-9]+", s.lower().replace(",", "")))'], 2: ['return f" {norm(ref)} " in f" {norm(answer)} "'], 3: ['calls = [int(contains(d["answer"], d["reference"])) for d in graded]', 'agree = np.mean([c == l for c, l in zip(calls, labels)])'], 4: ['po = np.mean(a == b); pa, pb = a.mean(), b.mean()', 'pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)'], 5: ['verdicts = [judge(d) for d in graded]', 'judge_kappa = kappa(verdicts, labels)'], 6: ['For each (q, right, wrong): a = prefer(q, right, wrong), b = prefer(q, wrong, right); consistent when a != b.', 'consistent = np.mean([prefer(q, r, w) != prefer(q, w, r) for q, r, w in pairs])'], 7: ['b = sum(x and not y for x, y in zip(plain_ok, rag_ok)); c the other way round', 'p = stats.binomtest(c, b + c, 0.5).pvalue']}
_ANSWERS = {1: 'cmV0dXJuICIgIi5qb2luKHJlLmZpbmRhbGwociJbYS16MC05XSsiLCBzLmxvd2VyKCkucmVwbGFjZSgiLCIsICIiKSkp', 2: 'cmV0dXJuIGYiIHtub3JtKHJlZil9ICIgaW4gZiIge25vcm0oYW5zd2VyKX0gIg==', 3: 'Y2FsbHMgPSBbaW50KGNvbnRhaW5zKGRbImFuc3dlciJdLCBkWyJyZWZlcmVuY2UiXSkpIGZvciBkIGluIGdyYWRlZF0KYWdyZWUgPSBucC5tZWFuKFtjID09IGwgZm9yIGMsIGwgaW4gemlwKGNhbGxzLCBsYWJlbHMpXSk=', 4: 'cG8gPSBucC5tZWFuKGEgPT0gYikKcGEsIHBiID0gYS5tZWFuKCksIGIubWVhbigpCnBlID0gcGEgKiBwYiArICgxIC0gcGEpICogKDEgLSBwYikKcmV0dXJuIChwbyAtIHBlKSAvICgxIC0gcGUp', 5: 'dmVyZGljdHMgPSBbanVkZ2UoZCkgZm9yIGQgaW4gZ3JhZGVkXQpqdWRnZV9rYXBwYSA9IGthcHBhKHZlcmRpY3RzLCBsYWJlbHMp', 6: 'Y29uc2lzdGVudCA9IG5wLm1lYW4oW3ByZWZlcihxLCByLCB3KSAhPSBwcmVmZXIocSwgdywgcikgZm9yIHEsIHIsIHcgaW4gcGFpcnNdKQ==', 7: 'YiA9IHN1bSh4IGFuZCBub3QgeSBmb3IgeCwgeSBpbiB6aXAocGxhaW5fb2ssIHJhZ19vaykpCmMgPSBzdW0oeSBhbmQgbm90IHggZm9yIHgsIHkgaW4gemlwKHBsYWluX29rLCByYWdfb2spKQpwID0gc3RhdHMuYmlub210ZXN0KGMsIGIgKyBjLCAwLjUpLnB2YWx1ZQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import json, re, urllib.request
import numpy as np
import torch
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
ev = json.load(urllib.request.urlopen(DATA + "llm-eval.json"))
graded, qa = ev["graded"], ev["qa"]
labels = [d["label"] for d in graded]
print(len(graded), "graded answers,", sum(labels), "of them correct ·", len(qa), "questions with accepted answers")
print(graded[0])

## 1. Normalize before comparing
Write `norm(s)`: lowercase, drop commas (so 65,504 = 65504), keep only runs of letters and digits
(`re.findall(r"[a-z0-9]+", ...)`), joined by single spaces.

In [ ]:
def norm(s):
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

In [ ]:
check(1, lambda: norm("ResNet-50 has 50 layers.") == "resnet 50 has 50 layers" and norm("65,504") == "65504", "lowercase words and numbers, punctuation gone")

## 2. A contains-check, and its agreement
Write `contains(answer, ref)`: true when the normalized reference appears as whole words in the normalized answer
(pad both with spaces: `f" {norm(ref)} " in f" {norm(answer)} "`). Then set `calls`, its 0/1 verdict on every graded
answer, and `agree`, the share that match `labels`.

In [ ]:
def contains(answer, ref):
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

calls = agree = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print("contains-check agrees on", agree)
for d, c in zip(graded, calls or []):
    if c != d["label"]:
        print("  disagrees:", d["answer"])

In [ ]:
check(3, lambda: abs(agree - 16 / 18) < 1e-9, "16 of 18: it's fooled by the two answers that mention the right word next to a wrong claim")

## 3. Cohen's kappa
Write `kappa(a, b)` for two lists of 0/1 verdicts: observed agreement `po`, chance agreement
`pe = pa * pb + (1 - pa) * (1 - pb)` where `pa`, `pb` are each grader's share of 1s, and `(po - pe) / (1 - pe)`.

In [ ]:
def kappa(a, b):
    a, b = np.array(a), np.array(b)
    # YOUR CODE HERE (stuck? hint(4), then answer(4))
    ...

print("contains-check kappa:", kappa(calls, labels) if calls is not None else None, "· always 'correct':", kappa([1] * len(labels), labels))

In [ ]:
check(4, lambda: abs(kappa([1] * 18, labels)) < 1e-9 and abs(kappa(labels, labels) - 1) < 1e-9 and abs(kappa(calls, labels) - 0.7778) < 1e-3,
      "1 for perfect agreement, 0 for the do-nothing grader, about 0.78 for the contains-check")

## 4. A small model as judge
`judge(item)` (given) asks SmolLM2-360M whether the answer is correct, given the question and reference. Set
`verdicts` to its 0/1 verdict on every graded answer, and `judge_kappa` to its kappa against `labels`.

In [ ]:
name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
def generate(prompt, n=40):
    ids = tok(tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True), return_tensors="pt").input_ids
    with torch.no_grad():
        return tok.decode(llm.generate(ids, max_new_tokens=n, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True)
def judge(item):
    reply = generate(f"Question: {item['q']}\nReference answer: {item['reference']}\nStudent answer: {item['answer']}\n"
                     "Is the student answer correct? Reply with only yes or no.", 5)
    return 1 if reply.strip().lower().startswith("yes") else 0

verdicts = judge_kappa = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print("judge says correct", sum(verdicts) if verdicts else None, "times · kappa", judge_kappa)

In [ ]:
check(5, lambda: len(verdicts) == 18 and judge_kappa < 0.5, "a weak judge: it agrees barely more than chance (kappa about 0.2). Validate judges before using them")

## 5. Position bias
Pairwise judging asks "which answer is better, 1 or 2?". `prefer(q, first, second)` (given) returns 1 or 2. For each
question below there's a right and a wrong answer. Ask both orders; the judge is *consistent* on a pair if it picks
the same answer both times, meaning 1 in one order and 2 in the other. Set `consistent` to the share of consistent
pairs.

In [ ]:
def prefer(q, first, second):
    reply = generate(f"Question: {q}\nAnswer 1: {first}\nAnswer 2: {second}\nWhich answer is correct? Reply with only 1 or 2.", 3)
    return 2 if "2" in reply else 1
pairs = [(d["q"], d["answer"], e["answer"]) for d in graded if d["label"] == 1 for e in graded if e["q"] == d["q"] and e["label"] == 0][:8]
consistent = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print(len(pairs), "pairs · consistent across orders:", consistent)

In [ ]:
check(6, lambda: 0 <= consistent <= 1 and len(pairs) == 8, "the share of pairs where swapping the order didn't change which answer won; anything under 1 is position bias")

## 6. Did retrieval help? A paired comparison
Both systems answer the same 20 questions (given: plain, and with the 3 best course passages by TF-IDF). Each answer
is graded with your `contains` against any accepted answer. Set `b` (questions only plain got right), `c` (only RAG
got right), and `p`: the exact paired test from Probability 4, `stats.binomtest(c, b + c, 0.5).pvalue`.

In [ ]:
idx = json.load(urllib.request.urlopen(DATA + "course-chunks.json")); sec = idx["sections"]
docs = [f"{s['lesson']}: {s['section']}. {s['text']}" for s in sec]
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit(docs); D = tfidf.transform(docs)
def rag_answer(q):
    top = np.argsort(-(tfidf.transform([q]) @ D.T).toarray()[0])[:3]
    ctx = "\n\n".join(f"[{k + 1}] {sec[i]['text'][:600]}" for k, i in enumerate(top))
    return generate(f"Use the passages to answer briefly.\n\n{ctx}\n\nQuestion: {q}")
right = lambda ans, accept: any(contains(ans, a) for a in accept)
plain_ok = [right(generate(f"Answer briefly: {x['q']}"), x["accept"]) for x in qa]
rag_ok = [right(rag_answer(x["q"]), x["accept"]) for x in qa]
print("plain:", sum(plain_ok), "/ 20 · with retrieval:", sum(rag_ok), "/ 20")
b = c = p = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print("only plain right:", b, "· only RAG right:", c, "· paired p =", p)

In [ ]:
check(7, lambda: b + c > 0 and abs(p - stats.binomtest(c, b + c, 0.5).pvalue) < 1e-12 and c > b,
      "retrieval wins every question that changed, and with 20 questions that's still not conclusive (p ≈ 0.1): eval sets need hundreds")

## Next
Back to the path: https://ihor.sh/ai/ · Anthropic, "Create strong empirical evaluations":
https://docs.anthropic.com/en/docs/test-and-evaluate/develop-tests